# Automate model building and deployment with a SageMaker project

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠ Compatibility Notice:</strong> This notebook has been tested using <strong>SageMaker Distribution Image 4.7.0</strong> and the <strong>SageMaker Python SDK version 3.29.0</strong>.
</div>

In the [SageMaker Pipelines lab](../sagemaker-pipelines/02_sagemaker-pipelines.ipynb) you ran an ML pipeline from a notebook and deployed the model by hand. In this notebook you move the same pipeline into a [SageMaker project](https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-projects.html), which runs it with CI/CD:

- a **model build** CI/CD pipeline runs the SageMaker pipeline on every commit to the model build repository and registers a new model version
- a **model deployment** CI/CD pipeline deploys every approved model version to a staging endpoint, tests it, waits for your approval, and deploys it to a production endpoint

You publish your own [project template](https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-projects-templates-custom.html) and seed code to Amazon S3 and create the project from them. The template is based on the SageMaker-provided template *MLOps template for model building, training, and deployment*, which used AWS CodeCommit repositories. SageMaker no longer offers that template, so you bring it yourself, together with seed code that uses the SageMaker Python SDK v3 and MLflow.

## What you do in this notebook

1. Explore the seed code of the two code repositories and the project template
2. Publish the template and the seed code to your S3 bucket
3. Create a SageMaker project from the template
4. Follow the model build: CodePipeline, CodeBuild, and the SageMaker pipeline with MLflow tracking
5. Approve the model version and follow the deployment: staging endpoint, endpoint test, your approval, production endpoint
6. Change the training code, push the change, and see the project build a new model version
7. Clean up

You don't need a GitHub account or a code connection. The project creates two AWS CodeCommit repositories and fills them with the seed code from S3.

<div class="alert alert-info"><b>Time and cost:</b> the model build takes about 15-20 minutes, each endpoint deployment about 5-10 minutes. The project creates CodeCommit, CodeBuild, CodePipeline, EventBridge, CloudFormation, and S3 resources, and two real-time endpoints. Run the <a href="#Clean-up">clean-up</a> section when you're done.</div>

## Architecture

The project's CloudFormation stack owns both CodeCommit repositories, the EventBridge rules, the CodeBuild projects, the two CodePipeline pipelines, and an S3 bucket for the pipeline artifacts.

**Model build**
1. A commit to the `main` branch of the `modelbuild` repository starts the model build pipeline.
2. CodeBuild runs `pipeline.py`. The script creates or updates the SageMaker pipeline `sagemaker-<project name>` and starts an execution.
3. The SageMaker pipeline preprocesses the data, trains an XGBoost model, evaluates it, and registers a model version with the status `PendingManualApproval` in the model package group `<project name>-<project ID>`. Every step logs to MLflow.

**Model deployment**
1. Approving a model version, or a commit to the `main` branch of the `modeldeploy` repository, starts the deployment pipeline.
2. CodeBuild selects the newest approved model version and writes the CloudFormation parameter files for staging and production.
3. CloudFormation creates the staging endpoint `<project name>-staging`. CodeBuild sends a test request to it.
4. The pipeline waits for your manual approval, then CloudFormation creates the production endpoint `<project name>-prod`.

---

## Setup
Install the packages of this lab. The cell is fast if they're already installed.

In [ ]:
import sys
!pip install -q uv
!uv pip install -q --python {sys.executable} -r ../requirements.txt

In [ ]:
import json
import os
import subprocess
import zipfile
from pathlib import Path
from time import gmtime, sleep, strftime

import boto3
import yaml
from botocore.exceptions import ClientError
from IPython.display import HTML, display
from sagemaker.core.helper.session_helper import Session

sagemaker_session = Session()
region = sagemaker_session.boto_region_name
account_id = boto3.client("sts").get_caller_identity()["Account"]
bucket_name = sagemaker_session.default_bucket()

sm = boto3.client("sagemaker")
s3 = boto3.client("s3")
iam = boto3.client("iam")
codepipeline = boto3.client("codepipeline")
codebuild = boto3.client("codebuild")
cfn = boto3.client("cloudformation")
sm_runtime = boto3.client("sagemaker-runtime")

print(f"Region: {region}")
print(f"Bucket: {bucket_name}")

Find the SageMaker domain of this JupyterLab space. The notebook uses it for the links to the Studio UI and to show the project template in Studio.

In [ ]:
NOTEBOOK_METADATA_FILE = "/opt/ml/metadata/resource-metadata.json"

domain_id = os.environ.get("SAGEMAKER_DOMAIN_ID")  # set this variable only if you run the notebook outside Studio
if os.path.exists(NOTEBOOK_METADATA_FILE):
    with open(NOTEBOOK_METADATA_FILE) as f:
        domain_id = json.load(f).get("DomainId")

print(f"SageMaker domain ID: {domain_id}")

### MLflow app
The model build pipeline logs to the SageMaker managed MLflow app `mlflow-app` of the workshop, the same app as in the previous labs.

In [ ]:
mlflow_app_name = "mlflow-app"

mlflow_app = next((a for a in sm.list_mlflow_apps()["Summaries"] if a["Name"] == mlflow_app_name), None)
if mlflow_app is None:
    raise Exception(f"The MLflow app {mlflow_app_name} doesn't exist. Create it in Studio under Applications > MLflow, or run the SageMaker Pipelines lab first.")

mlflow_arn = mlflow_app["Arn"]
while sm.describe_mlflow_app(Arn=mlflow_arn)["Status"] not in ["Created", "Updated"]:
    print("Waiting for the MLflow app...")
    sleep(10)

print(f"MLflow app: {mlflow_arn}")

### Container image of the pipeline steps
In Studio, the `@step` functions of the SageMaker Pipelines lab run on the image of your JupyterLab space by default. In the project, CodeBuild builds the pipeline, so you pass the image explicitly. The pipeline steps use the image of this space.

CodeBuild runs Python 3.12, and the `@step` decorator needs the same Python version in the image. The next cell checks it.

In [ ]:
pipeline_step_image_uri = os.environ.get("SAGEMAKER_INTERNAL_IMAGE_URI")
assert pipeline_step_image_uri, "Run this notebook in a JupyterLab space in SageMaker Studio, or set SAGEMAKER_INTERNAL_IMAGE_URI to a SageMaker Distribution image URI"
assert sys.version_info[:2] == (3, 12), (
    f"The image runs Python {sys.version_info.major}.{sys.version_info.minor}, the model build CodeBuild project runs Python 3.12. "
    "Use SageMaker Distribution 4.7.0 for this space."
)

print(f"Pipeline step image: {pipeline_step_image_uri}")

---

## Explore the seed code
The seed code is the initial content of the two code repositories of the project. Both are regular folders in this lab: you own them, version them, and review them like any other code. When you create the project, CloudFormation copies each folder from a ZIP file in S3 into a new CodeCommit repository. From then on, the repositories are the source of the CI/CD pipelines.

**[`seed-code/model-build`](seed-code/model-build)**: the model build repository

|File|Purpose|
|---|---|
|`codebuild-buildspec.yml`|CodeBuild instructions: install `requirements.txt` and run `pipeline.py`|
|`pipeline.py`|Builds the SageMaker pipeline from the step functions with the `@step` decorator, upserts it, and starts an execution|
|`steps/`|The step functions `preprocess`, `train`, `evaluate`, and `register` of the SageMaker Pipelines lab|
|`requirements.txt`|The packages that CodeBuild and every pipeline step install|

**[`seed-code/model-deploy`](seed-code/model-deploy)**: the model deployment repository

|File|Purpose|
|---|---|
|`buildspec.yml`, `build.py`|Select the newest `Approved` model version and write the CloudFormation parameter files for staging and production|
|`endpoint-config-template.yml`|CloudFormation template with the SageMaker model, the endpoint configuration with data capture, and the endpoint|
|`staging-config.json`, `prod-config.json`|Stage settings: instance type and count, data capture sampling percentage|
|`test/buildspec.yml`, `test/test.py`, `test/payload.csv`|The staging test: one real inference request with a preprocessed abalone record|

[`ORIGIN.md`](ORIGIN.md) lists where the seed code and the template come from and what changed.

In [ ]:
seed_code_directory = Path("seed-code")

for path in sorted(seed_code_directory.rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts and ".ipynb_checkpoints" not in path.parts:
        print(path)

The core of the model build is `pipeline.py`. It chains the same step functions as in the SageMaker Pipelines lab. The CodeBuild project passes the pipeline name, the role, the model package group, the MLflow app, and the image as arguments.

In [ ]:
!sed -n '/def main/,$p' seed-code/model-build/pipeline.py

### Package the seed code
CodeCommit expands the root of the ZIP file into the root of the new repository. The next cell packages each seed folder at the archive root and skips local artifacts such as `.ipynb_checkpoints`.

In [ ]:
build_directory = Path("tmp")
build_directory.mkdir(exist_ok=True)


def package_seed_code(source_directory, zip_name):
    archive_path = build_directory / zip_name
    files = sorted(
        p for p in source_directory.rglob("*")
        if p.is_file()
        and not {"__pycache__", ".ipynb_checkpoints"}.intersection(p.parts)
        and p.suffix != ".pyc"
    )
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in files:
            archive.write(path, path.relative_to(source_directory).as_posix())
    print(f"{archive_path}: {len(files)} files, {archive_path.stat().st_size:,} bytes")
    return archive_path


model_build_zip = package_seed_code(seed_code_directory / "model-build", "model-build-seed-code.zip")
model_deploy_zip = package_seed_code(seed_code_directory / "model-deploy", "model-deploy-seed-code.zip")

---

## Explore the project template
The project template [`project-template/build-train-deploy-codecommit.yaml`](project-template/build-train-deploy-codecommit.yaml) is a CloudFormation template. SageMaker sets the parameters `SageMakerProjectName` and `SageMakerProjectId` when it creates a project. You provide the other parameters:

- `SeedCodeBucket`, `ModelBuildSeedCodeKey`, `ModelDeploySeedCodeKey`: the S3 location of the two seed code ZIP files
- `MlflowTrackingServerArn`: the MLflow app that the model build logs to
- `PipelineStepImageUri`: the container image of the pipeline steps

The next cell lists the resources of the template and the stages of the two CI/CD pipelines.

In [ ]:
project_template_path = Path("project-template/build-train-deploy-codecommit.yaml")


# CloudFormation short-form functions like !Ref and !Sub are not standard YAML tags, so ignore them for this overview
class CfnLoader(yaml.SafeLoader):
    pass


CfnLoader.add_multi_constructor("!", lambda loader, suffix, node: None)
project_template = yaml.load(project_template_path.read_text(), Loader=CfnLoader)

print(f"Parameters: {', '.join(project_template['Parameters'])}\n")
for logical_id, resource in project_template["Resources"].items():
    print(f"{resource['Type']:<32} {logical_id}")

for pipeline in ["ModelBuildPipeline", "ModelDeployPipeline"]:
    stages = [s["Name"] for s in project_template["Resources"][pipeline]["Properties"]["Stages"]]
    print(f"\n{pipeline}: {' -> '.join(stages)}")

### IAM roles
The project uses two roles that the workshop environment created:

- **Launch role** `AmazonSageMakerServiceCatalogProductsLaunchRole`: CloudFormation uses it to create the project resources, such as the repositories, the CodeBuild projects, and the pipelines. You only need the permission to pass this role, not the permissions to create the resources yourself.
- **Use role** `AmazonSageMakerServiceCatalogProductsUseRole`: EventBridge, CodePipeline, CodeBuild, CloudFormation, the SageMaker pipeline, and the endpoints use it when the project runs.

In [ ]:
launch_role_arn = f"arn:aws:iam::{account_id}:role/service-role/AmazonSageMakerServiceCatalogProductsLaunchRole"
use_role_arn = f"arn:aws:iam::{account_id}:role/service-role/AmazonSageMakerServiceCatalogProductsUseRole"

print(f"Launch role: {launch_role_arn}")
print(f"Use role:    {use_role_arn}")

---

## Publish the template and the seed code to S3
SageMaker creates projects from templates in S3. The next cell uploads the seed code ZIP files and the template to your bucket. The template object gets the tag `sagemaker:studio-visibility=true`, so that Studio shows it in the list of organization templates.

In [ ]:
s3_prefix = "sagemaker-projects-lab"
template_key = f"{s3_prefix}/templates/build-train-deploy-codecommit.yaml"
model_build_seed_code_key = f"{s3_prefix}/seed-code/{model_build_zip.name}"
model_deploy_seed_code_key = f"{s3_prefix}/seed-code/{model_deploy_zip.name}"

s3.upload_file(str(model_build_zip), bucket_name, model_build_seed_code_key)
s3.upload_file(str(model_deploy_zip), bucket_name, model_deploy_seed_code_key)
s3.upload_file(str(project_template_path), bucket_name, template_key)
s3.put_object_tagging(
    Bucket=bucket_name,
    Key=template_key,
    Tagging={"TagSet": [{"Key": "sagemaker:studio-visibility", "Value": "true"}]},
)

template_url = f"https://{bucket_name}.s3.{region}.amazonaws.com/{template_key}"
print(f"Model build seed code:      s3://{bucket_name}/{model_build_seed_code_key}")
print(f"Model deployment seed code: s3://{bucket_name}/{model_deploy_seed_code_key}")
print(f"Template:                   {template_url}")

### Show the template in Studio
To list S3 templates in the Studio UI, the SageMaker domain needs the tag `sagemaker:projectS3TemplatesLocation` with the S3 location of the templates, and the bucket needs a CORS policy that allows Studio to read them. You need this only if you create the project in the Studio UI (Option 2 below).

In [ ]:
STUDIO_TEMPLATES_CORS_RULE = {
    "ID": "sagemaker-projects-lab-studio-templates",
    "AllowedHeaders": ["*"],
    "AllowedMethods": ["GET", "HEAD", "PUT", "POST"],
    "AllowedOrigins": ["https://*.sagemaker.aws"],
    "ExposeHeaders": ["ETag", "x-amz-delete-marker", "x-amz-id-2", "x-amz-request-id", "x-amz-server-side-encryption", "x-amz-version-id"],
}

if domain_id:
    domain_arn = sm.describe_domain(DomainId=domain_id)["DomainArn"]
    sm.add_tags(
        ResourceArn=domain_arn,
        Tags=[{"Key": "sagemaker:projectS3TemplatesLocation", "Value": f"s3://{bucket_name}/{s3_prefix}/templates/"}],
    )
    # Add the Studio rule to the existing CORS rules of the bucket, for example the rule for Canvas uploads
    try:
        cors_rules = s3.get_bucket_cors(Bucket=bucket_name)["CORSRules"]
    except ClientError:
        cors_rules = []
    if not any(r.get("ID") == STUDIO_TEMPLATES_CORS_RULE["ID"] for r in cors_rules):
        s3.put_bucket_cors(Bucket=bucket_name, CORSConfiguration={"CORSRules": cors_rules + [STUDIO_TEMPLATES_CORS_RULE]})
    print(f"Studio lists the templates in s3://{bucket_name}/{s3_prefix}/templates/")
else:
    print("No domain found, skip the Studio UI setup")

---

## Create a SageMaker project
You can create a project programmatically in this notebook - **Option 1** - or in the Studio UI - **Option 2**.

Option 1 is recommended as it has no dependency on the UI.

### Option 1: Create a project programmatically - recommended
The [`CreateProject`](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_CreateProject.html) API with a [`CfnTemplateProvider`](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_CfnTemplateProviderDetail.html) reads the template directly from S3. You don't need AWS Service Catalog. The project name must be unique in your account and Region, and have at most 32 characters.

In [ ]:
project_name = f"mlops-{strftime('%d-%H-%M-%S', gmtime())}"

r = sm.create_project(
    ProjectName=project_name,
    ProjectDescription="Build, train, and deploy the abalone model with CodeCommit, CodeBuild, and CodePipeline",
    TemplateProviders=[
        {
            "CfnTemplateProvider": {
                "TemplateName": "BuildTrainDeployCodeCommit",
                "TemplateURL": template_url,
                "RoleARN": launch_role_arn,
                "Parameters": [
                    {"Key": "SeedCodeBucket", "Value": bucket_name},
                    {"Key": "ModelBuildSeedCodeKey", "Value": model_build_seed_code_key},
                    {"Key": "ModelDeploySeedCodeKey", "Value": model_deploy_seed_code_key},
                    {"Key": "MlflowTrackingServerArn", "Value": mlflow_arn},
                    {"Key": "PipelineStepImageUri", "Value": pipeline_step_image_uri},
                ],
            }
        }
    ],
)
print(f"Creating the project {r['ProjectArn']}")

<div class="alert alert-info">Wait until the project creation is completed by running the next cell. It takes 1-2 minutes.</div>

In [ ]:
status = sm.describe_project(ProjectName=project_name)["ProjectStatus"]
while status in ["Pending", "CreateInProgress"]:
    print("Waiting for project creation completion")
    sleep(10)
    status = sm.describe_project(ProjectName=project_name)["ProjectStatus"]

project_data = sm.describe_project(ProjectName=project_name)
print(f"Project {project_name} status: {status} {project_data.get('ProjectStatusMessage', '')}")
assert status == "CreateCompleted", "Project status must be CreateCompleted!"

### End of Option 1: Create a project programmatically
Continue with the section **Explore the project**.

---

### Option 2: Create a project in the Studio UI
<div class="alert alert-info"><b>Skip this section if you have already created a project programmatically</b></div>

1. In Studio, choose **Deployments** > **Projects** > **Create project**
2. Choose **Organization templates** > **S3 Templates**
3. Select the template `build-train-deploy-codecommit` and choose **Next**
4. Enter a project name and the template parameters printed by the next cell. In the **Role ARN** field, enter the ARN of the launch role.
5. Choose **Create** and wait until the project is created

In [ ]:
print(f"Seed code S3 bucket:               {bucket_name}")
print(f"Model build seed code S3 key:      {model_build_seed_code_key}")
print(f"Model deployment seed code S3 key: {model_deploy_seed_code_key}")
print(f"MLflow app ARN:                    {mlflow_arn}")
print(f"Pipeline step image URI:           {pipeline_step_image_uri}")
print(f"Role ARN:                          {launch_role_arn}")

Set the variable `project_name` in the next cell to the name of the project you created in the Studio UI and uncomment the line.

In [ ]:
# project_name = "<SET TO THE NAME OF THE CREATED PROJECT>" # Keep commented out if you used option 1 to create a project

### End of Option 2: Create a project in the Studio UI

---

## Explore the project
The project names all resources after the project name and the project ID.

In [ ]:
project_data = sm.describe_project(ProjectName=project_name)
assert project_data["ProjectStatus"] == "CreateCompleted", "Project status must be CreateCompleted!"
project_id = project_data["ProjectId"]

resource_prefix = f"sagemaker-{project_name}-{project_id}"
build_pipeline_name = f"{resource_prefix}-modelbuild"
deploy_pipeline_name = f"{resource_prefix}-modeldeploy"
build_repository_name = f"{resource_prefix}-modelbuild"
deploy_repository_name = f"{resource_prefix}-modeldeploy"
sagemaker_pipeline_name = f"sagemaker-{project_name}"
model_package_group_name = f"{project_name}-{project_id}"
staging_endpoint_name = f"{project_name}-staging"
prod_endpoint_name = f"{project_name}-prod"

print(f"Project:                  {project_name} ({project_id})")
print(f"Model build repository:   {build_repository_name}")
print(f"Model build pipeline:     {build_pipeline_name}")
print(f"SageMaker pipeline:       {sagemaker_pipeline_name}")
print(f"Model package group:      {model_package_group_name}")
print(f"Deployment repository:    {deploy_repository_name}")
print(f"Deployment pipeline:      {deploy_pipeline_name}")
print(f"Endpoints:                {staging_endpoint_name}, {prod_endpoint_name}")

Click on the links constructed by the next cell to see the project in the Studio UI, the two CI/CD pipelines in the CodePipeline console, and the repositories in the CodeCommit console.

In [ ]:
def link(text, url):
    display(HTML(f'<b>See <a target="_blank" rel="noopener noreferrer" href="{url}">{text}</a></b>'))


console = "https://console.aws.amazon.com"
if domain_id:
    link("the project in the Studio UI", f"https://studio-{domain_id}.studio.{region}.sagemaker.aws/projects/{project_name}/")
for name in [build_pipeline_name, deploy_pipeline_name]:
    link(f"the CI/CD pipeline {name}", f"{console}/codesuite/codepipeline/pipelines/{name}/view?region={region}")
for name in [build_repository_name, deploy_repository_name]:
    link(f"the repository {name}", f"{console}/codesuite/codecommit/repositories/{name}/browse?region={region}")

---

## Model build
CodePipeline starts both CI/CD pipelines right after the project creates them:

- The **model build pipeline** runs CodeBuild, which creates the SageMaker pipeline and starts an execution.
- The **deployment pipeline** fails in the `Build` stage, because the model package group has no approved model version yet. This is expected. The approval of the first model version starts it again.

The next cell waits until the CodeBuild run of the model build pipeline completes. It takes about 3 minutes, most of it to install the packages.

In [ ]:
def latest_pipeline_execution(pipeline_name):
    executions = codepipeline.list_pipeline_executions(pipelineName=pipeline_name, maxResults=1)["pipelineExecutionSummaries"]
    return executions[0] if executions else None


def pipeline_failure(pipeline_name):
    """Return a description of the first failed action of the latest execution or None."""
    execution = latest_pipeline_execution(pipeline_name)
    if not execution or execution["status"] not in ["Failed", "Stopped", "Superseded"]:
        return None
    for stage in codepipeline.get_pipeline_state(name=pipeline_name)["stageStates"]:
        for action in stage["actionStates"]:
            latest = action.get("latestExecution", {})
            if latest.get("status") == "Failed":
                message = latest.get("errorDetails", {}).get("message", "")
                return f"{stage['stageName']}/{action['actionName']} failed: {message}\nLogs: {latest.get('externalExecutionUrl', '')}"
    return f"Pipeline execution is {execution['status']}"


def wait_for_pipeline_execution(pipeline_name, after_execution_id=None):
    """Wait for a new execution, if after_execution_id is set, and until the latest execution completes."""
    while True:
        execution = latest_pipeline_execution(pipeline_name)
        if execution and execution["pipelineExecutionId"] != after_execution_id and execution["status"] not in ["InProgress", "Stopping"]:
            break
        print(f"{strftime('%H:%M:%S', gmtime())} {pipeline_name}: {execution['status'] if execution else 'not started'}")
        sleep(30)
    failure = pipeline_failure(pipeline_name)
    if failure:
        raise Exception(failure)
    print(f"{pipeline_name}: {execution['status']}")
    return execution


build_execution = wait_for_pipeline_execution(build_pipeline_name)

CodeBuild ran `codebuild-buildspec.yml`. The next cell shows the end of the CodeBuild log: `pipeline.py` uploaded the step functions, upserted the SageMaker pipeline, and started an execution.

In [ ]:
logs = boto3.client("logs")
build_id = codebuild.list_builds_for_project(projectName=build_pipeline_name)["ids"][0]
build = codebuild.batch_get_builds(ids=[build_id])["builds"][0]
log_events = logs.get_log_events(
    logGroupName=build["logs"]["groupName"], logStreamName=build["logs"]["streamName"], startFromHead=False, limit=40
)["events"]
print("".join(e["message"] for e in log_events if "Running pip as the 'root' user" not in e["message"])[-3000:])
link("the full CodeBuild log", build["logs"]["deepLink"])

### Follow the SageMaker pipeline
The SageMaker pipeline has four steps: preprocessing, training, evaluation, and registration. It takes about 15 minutes. The pipeline is tagged with the project name and ID, so you also find it in the Studio UI under the project, on the **Pipelines** tab.

In [ ]:
def latest_sagemaker_pipeline_execution():
    executions = sm.list_pipeline_executions(PipelineName=sagemaker_pipeline_name, SortBy="CreationTime", SortOrder="Descending")
    return executions["PipelineExecutionSummaries"][0]


sagemaker_execution_arn = latest_sagemaker_pipeline_execution()["PipelineExecutionArn"]
if domain_id:
    link("the pipeline execution in the Studio UI",
         f"https://studio-{domain_id}.studio.{region}.sagemaker.aws/pipelines/{sagemaker_pipeline_name}/executions/{sagemaker_execution_arn.split('/')[-1]}/graph")


def wait_for_sagemaker_pipeline(execution_arn):
    while True:
        status = sm.describe_pipeline_execution(PipelineExecutionArn=execution_arn)["PipelineExecutionStatus"]
        steps = sm.list_pipeline_execution_steps(PipelineExecutionArn=execution_arn, SortOrder="Ascending")["PipelineExecutionSteps"]
        print(f"{strftime('%H:%M:%S', gmtime())} {status}: " + ", ".join(f"{s['StepName']} {s['StepStatus']}" for s in steps))
        if status != "Executing":
            break
        sleep(60)
    for s in steps:
        if s["StepStatus"] == "Failed":
            raise Exception(f"{s['StepName']} failed: {s.get('FailureReason', '')}")
    return status


wait_for_sagemaker_pipeline(sagemaker_execution_arn)

### See the runs in MLflow
`pipeline.py` creates a parent run in the MLflow experiment `sagemaker-<project name>` for each version of the pipeline, and each step logs a nested run to it: the dataset, the training metrics and the model, the test MSE, and the model package ARN. Open the MLflow UI and choose the experiment.

In [ ]:
presigned_url = sm.create_presigned_mlflow_app_url(Arn=mlflow_arn, ExpiresInSeconds=300, SessionExpirationDurationInSeconds=1800)["AuthorizedUrl"]
display(HTML(f'<b><a target="_blank" rel="noopener noreferrer" href="{presigned_url}">Open the MLflow UI</a></b> and choose the experiment <code>sagemaker-{project_name}</code>'))

---

## Approve the model version
The registration step added a model version with the status `PendingManualApproval` to the model package group. The model package group has the project tags, so you see it in the Studio UI under the project, on the **Model groups** tab.

You can approve the model version in the Studio UI: open the model group, select the version, choose **Update status**, and set it to **Approved**. Or run the next cell, which approves the newest version.

In [ ]:
model_package = sm.list_model_packages(
    ModelPackageGroupName=model_package_group_name, SortBy="CreationTime", SortOrder="Descending", MaxResults=1
)["ModelPackageSummaryList"][0]
print(f"Newest model version: {model_package['ModelPackageArn']} ({model_package['ModelApprovalStatus']})")

if domain_id:
    link("the model versions in the Studio UI",
         f"https://studio-{domain_id}.studio.{region}.sagemaker.aws/registry/registered-models/{model_package_group_name}/versions")

In [ ]:
deploy_execution_before_approval = latest_pipeline_execution(deploy_pipeline_name)

if model_package["ModelApprovalStatus"] != "Approved":
    sm.update_model_package(ModelPackageArn=model_package["ModelPackageArn"], ModelApprovalStatus="Approved")
    print(f"Approved {model_package['ModelPackageArn']}")

---

## Model deployment
The approval fires an EventBridge rule that starts the deployment pipeline:
1. `Source`: get the code from the `main` branch of the deployment repository
1. `Build`: select the newest approved model version and create the CloudFormation parameter files for staging and production
1. `DeployStaging`: create the staging endpoint `<project name>-staging`, run the test script on it, and wait for manual approval
1. `DeployProd`: create the production endpoint `<project name>-prod`

The staging deployment takes about 5-10 minutes. The next cell waits until the staging endpoint is `InService`.

In [ ]:
def wait_for_endpoint(endpoint_name):
    while True:
        try:
            status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
        except ClientError:
            status = "NotCreated"
        if status == "InService":
            break
        if status == "Failed":
            raise Exception(sm.describe_endpoint(EndpointName=endpoint_name).get("FailureReason"))
        execution = latest_pipeline_execution(deploy_pipeline_name)
        if execution and execution["pipelineExecutionId"] != deploy_execution_before_approval["pipelineExecutionId"]:
            failure = pipeline_failure(deploy_pipeline_name)
            if failure:
                raise Exception(failure)
        print(f"{strftime('%H:%M:%S', gmtime())} Endpoint {endpoint_name}: {status}")
        sleep(30)
    print(f"Endpoint {endpoint_name} is InService")


wait_for_endpoint(staging_endpoint_name)

### Test the staging endpoint
The `TestStaging` action of the pipeline sends one record to the staging endpoint. Send the same record from the notebook. The model predicts the number of rings of the abalone. The record is preprocessed: scaled numeric features and the one-hot encoded `sex`.

In [ ]:
payload = Path("seed-code/model-deploy/test/payload.csv").read_text().strip()

response = sm_runtime.invoke_endpoint(EndpointName=staging_endpoint_name, ContentType="text/csv", Accept="text/csv", Body=payload)
print(f"Input:           {payload}")
print(f"Predicted rings: {response['Body'].read().decode('utf-8').strip()}")

---

## Deploy the model version to production
After the `TestStaging` action succeeds, the pipeline waits in the `ApproveDeployment` action. Approving this action deploys the model to the production endpoint. You can approve in the CodePipeline console or programmatically.

<div class="alert alert-info">The production endpoint is a second billable endpoint.</div>

### Option 1: Approve in the CodePipeline console
Click on the link constructed by the next cell. In the **DeployStaging** stage, wait until `TestStaging` has the status `Succeeded`, and choose **Review** on the **ApproveDeployment** action. In the **Review** dialog box, select **Approve** and choose **Submit**.

In [ ]:
link("the deployment pipeline in the CodePipeline console", f"{console}/codesuite/codepipeline/pipelines/{deploy_pipeline_name}/view?region={region}")

### Option 2: Approve programmatically
The next cell waits until the approval action is active, and then approves it. If you already approved in the console, the cell does nothing.

In [ ]:
def approval_action_state():
    stage = next(s for s in codepipeline.get_pipeline_state(name=deploy_pipeline_name)["stageStates"] if s["stageName"] == "DeployStaging")
    action = next(a for a in stage["actionStates"] if a["actionName"] == "ApproveDeployment")
    return action.get("latestExecution", {})


while True:
    approval = approval_action_state()
    if approval.get("status") in ["InProgress", "Succeeded"]:
        break
    failure = pipeline_failure(deploy_pipeline_name)
    if failure:
        raise Exception(failure)
    print(f"{strftime('%H:%M:%S', gmtime())} Waiting for the ApproveDeployment action")
    sleep(20)

if approval["status"] == "Succeeded":
    print("The production deployment is already approved")
else:
    codepipeline.put_approval_result(
        pipelineName=deploy_pipeline_name,
        stageName="DeployStaging",
        actionName="ApproveDeployment",
        result={"summary": "Staging endpoint tested in the notebook", "status": "Approved"},
        token=approval["token"],
    )
    print("Approved the production deployment")

The production deployment takes about 5-10 minutes. The next cell waits until the production endpoint is `InService`, and sends the test record to it.

In [ ]:
wait_for_endpoint(prod_endpoint_name)

response = sm_runtime.invoke_endpoint(EndpointName=prod_endpoint_name, ContentType="text/csv", Accept="text/csv", Body=payload)
print(f"Predicted rings: {response['Body'].read().decode('utf-8').strip()}")

### Inspect the lineage
Both endpoints run the same approved model version. In the Studio UI, open the model version and choose the **Lineage** tab: it links the datasets, the jobs of the SageMaker pipeline, the model version, its approval, and the endpoints.

---

## Change the model and let the project rebuild it
The repositories are now the source of truth. A data scientist changes the model by pushing to the model build repository, not by running a notebook. In this section you clone the model build repository, change a hyperparameter, and push the change. The push starts the model build pipeline, which registers a new model version.

The next cell clones the repository with the CodeCommit credential helper, which uses the IAM role of this space.

In [ ]:
clone_directory = build_directory / build_repository_name
git_credentials = ["-c", "credential.helper=!aws codecommit credential-helper $@", "-c", "credential.UseHttpPath=true"]


def git(*args, cwd=clone_directory):
    result = subprocess.run(["git", *git_credentials, *args], cwd=cwd, capture_output=True, text=True)
    print(result.stdout + result.stderr)
    result.check_returncode()


if not clone_directory.exists():
    git("clone", f"https://git-codecommit.{region}.amazonaws.com/v1/repos/{build_repository_name}", str(clone_directory), cwd=".")
git("log", "--oneline", "-3")

Change the number of boosting rounds in `steps/train.py` from 50 to 100.

In [ ]:
train_file = clone_directory / "steps/train.py"
train_code = train_file.read_text()
assert "num_round=50," in train_code, "num_round is not 50 anymore"
train_file.write_text(train_code.replace("num_round=50,", "num_round=100,"))

git("diff")

In [ ]:
build_execution_before_push = latest_pipeline_execution(build_pipeline_name)

git("-c", "user.name=SageMaker Immersion Day", "-c", "user.email=participant@example.com", "commit", "-am", "Train with 100 boosting rounds")
git("push", "origin", "main")

The push starts the model build pipeline. The next cell waits until CodeBuild has started the SageMaker pipeline with the new code. You can follow the pipeline in the Studio UI, or run the cell after the next one to wait for the new model version. That takes another 15 minutes, so skip it if you're short on time.

In [ ]:
wait_for_pipeline_execution(build_pipeline_name, after_execution_id=build_execution_before_push["pipelineExecutionId"])
sagemaker_execution_arn = latest_sagemaker_pipeline_execution()["PipelineExecutionArn"]
print(f"SageMaker pipeline execution: {sagemaker_execution_arn}")

In [ ]:
# Optional: wait for the new model version, about 15 minutes
wait_for_sagemaker_pipeline(sagemaker_execution_arn)

for p in sm.list_model_packages(ModelPackageGroupName=model_package_group_name, SortBy="CreationTime", SortOrder="Descending")["ModelPackageSummaryList"]:
    print(f"Version {p['ModelPackageVersion']}: {p['ModelApprovalStatus']}")

To deploy the new version, approve it as before. The deployment pipeline updates the staging endpoint, tests it, and waits for your approval to update the production endpoint.

---

## Clean up
The next cell deletes everything that this notebook created, in this order:
1. the staging and production endpoint stacks that the deployment pipeline created. The stacks own the endpoints, the endpoint configurations, and the models.
2. the project. SageMaker deletes the project stack with the repositories, the CodeBuild projects, the CI/CD pipelines, and the EventBridge rules.
3. the S3 bucket `sagemaker-project-<project ID>` with the pipeline artifacts and the captured data. The project stack keeps the bucket when it's deleted.
4. the SageMaker pipeline and the model package group with all model versions, which CodeBuild created
5. the template, the seed code, and the Studio UI settings of this notebook

The runs in MLflow stay. Delete the experiment `sagemaker-<project name>` in the MLflow UI if you want.

In [ ]:
def delete_stack(stack_name):
    try:
        cfn.describe_stacks(StackName=stack_name)
    except ClientError:
        return
    print(f"Deleting the stack {stack_name}")
    cfn.delete_stack(StackName=stack_name)
    cfn.get_waiter("stack_delete_complete").wait(StackName=stack_name, WaiterConfig={"Delay": 15, "MaxAttempts": 120})


def delete_bucket(name):
    try:
        bucket = boto3.resource("s3").Bucket(name)
        bucket.object_versions.delete()
        bucket.delete()
        print(f"Deleted the bucket {name}")
    except ClientError as e:
        print(f"[WARN] {name}: {e.response['Error']['Message']}")


# 1. Endpoint stacks
for stage in ["deploy-staging", "deploy-prod"]:
    delete_stack(f"{resource_prefix}-{stage}")

# 2. The project and its stack
sm.delete_project(ProjectName=project_name)
while True:
    try:
        status = sm.describe_project(ProjectName=project_name)["ProjectStatus"]
    except ClientError:
        break
    if status in ["DeleteCompleted", "DeleteFailed"]:
        break
    print(f"Project {project_name}: {status}")
    sleep(15)
print(f"Deleted the project {project_name}")

# 3. The project bucket
delete_bucket(f"sagemaker-project-{project_id}")

# 4. The SageMaker pipeline and the model package group
try:
    sm.delete_pipeline(PipelineName=sagemaker_pipeline_name)
    print(f"Deleted the pipeline {sagemaker_pipeline_name}")
except ClientError as e:
    print(f"[WARN] {e.response['Error']['Message']}")

for p in sm.list_model_packages(ModelPackageGroupName=model_package_group_name)["ModelPackageSummaryList"]:
    sm.delete_model_package(ModelPackageName=p["ModelPackageArn"])
for _ in range(24):  # deleting a model version is asynchronous
    if not sm.list_model_packages(ModelPackageGroupName=model_package_group_name)["ModelPackageSummaryList"]:
        break
    sleep(5)
sm.delete_model_package_group(ModelPackageGroupName=model_package_group_name)
print(f"Deleted the model package group {model_package_group_name}")

# 5. Template, seed code, step artifacts of the pipeline, and Studio UI settings
boto3.resource("s3").Bucket(bucket_name).objects.filter(Prefix=f"{s3_prefix}/").delete()
boto3.resource("s3").Bucket(bucket_name).objects.filter(Prefix=f"{sagemaker_pipeline_name}/").delete()
if domain_id:
    sm.delete_tags(ResourceArn=sm.describe_domain(DomainId=domain_id)["DomainArn"], TagKeys=["sagemaker:projectS3TemplatesLocation"])
    # Remove only the CORS rule of this notebook, keep the other rules of the bucket
    try:
        other_rules = [r for r in s3.get_bucket_cors(Bucket=bucket_name)["CORSRules"] if r.get("ID") != STUDIO_TEMPLATES_CORS_RULE["ID"]]
        if other_rules:
            s3.put_bucket_cors(Bucket=bucket_name, CORSConfiguration={"CORSRules": other_rules})
        else:
            s3.delete_bucket_cors(Bucket=bucket_name)
    except ClientError:
        pass
print("Deleted the template, the seed code, and the Studio UI settings")

---

## Summary
In this notebook you:
- published your own SageMaker project template and seed code to S3, and created a project from it without AWS Service Catalog
- ran the SageMaker pipeline of the previous lab from a CI/CD pipeline, with MLflow tracking
- deployed an approved model version to a staging endpoint, tested it, and promoted it to production after a manual approval
- changed the training code with a `git push`, which built and registered a new model version

## Further development ideas
- Add a quality gate to the SageMaker pipeline: a condition step that registers the model only if its test MSE is below a threshold
- Add more tests to the staging stage, for example a load test or a comparison with the production endpoint
- Use separate AWS accounts for staging and production, with cross-account deployments from the model registry
- Add [model monitoring](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor.html) to the endpoints in `endpoint-config-template.yml`